# Práctica 3 - Ejercicio 2

Asignatura: Programación para la Inteligencia Artificial

Alumno: Fernández Roldán, Daniel

Uno de los primeros modelos propuestos para la generación de datos usando Aprendizaje profundo fue el *Variational Autoencoder* (VAE).

Un VAE es una red autocodificadora que durante su entrenamiento, en lugar de codificar a un espacio latente de manera determinista (cada ejemplo tiene asociada una codificación), asocia distribuciones gaussianas que se muestrean para obtener una codificación que seguidamente se decodifica. Por tanto, el proceso queda como sigue:

$$\mu, \sigma^2 = e_{\theta_1}(x)$$

$$z \sim \mathcal{N}(\mu, \sigma^2) $$

$$y = d_{\theta_{2}} (z)$$

Para realizar el entrenamiento como cualquier red autocodificadora, se incluye una función de pérdida de reconstrucción:

$$L_{r}(x,y)$$

Pero además se añade una función de pérdida con el objetivo de regularizar las distribuciones gaussianas que pueden aprenderse. Para esto se usa la Divergencia de Kullback-Leibler, que mide el grado de disimilitud entre una distribución de probabilidad $Q$ respecto a una distribución de probabilidad $P$:

$$D_{KL}(P,Q) = \sum_{x \in X} P(x) log (\frac{P(x)}{Q(x)})$$

La divergencia KL se utiliza para penalizar que las distribuciones gaussianas se alejen de la distribución gaussiana de media 0 y varianza 1 ($\mathcal{N}(0, 1)$). Teniendo eso en cuenta, su formulación se puede simplificar a:

$$L_{KL}(\mu, \sigma)= \sum_{i=0}^n -\frac{1}{2}(1+log (\sigma_i^2) - \mu_i^2 - \sigma_i^2)$$

Siendo la función de pérdida que se usa durante el entrenamiento:

$$L = L_{r} + \alpha L_{KL}$$

con $\alpha$ un hiperparámetro que controla cuánto peso tiene la divergencia de Kullback-Leibler respecto a la pérdida de reconstrucción.

Tras implementar el VAE, se pide entrenarlo para los datos de  Fashion-MNIST con un espacio latente de 8 dimensiones (el vector $\mu$ y el que representa sus respectivas varianzas tienen 8 dimensiones cada uno). Una vez entrenado y comprobado debidamente que funciona, se pide mostrar las componentes del vector $\mu$ del conjunto de test en el espacio latente usando gráficas bidimensionales que indican la clase de cada ejemplo como en el Ejercicio 1.

Después, se debe extraer el vector prototípico para cada clase en el conjunto de test (la media de los vectores $\mu$ para cada clase) y mostrar las imágenes generadas por el decodificador a partir de las codificaciones interpoladas entre los vectores prototípicos de dos clases cualesquiera.

El cuaderno entregado debe llamarse ApellidosNombrePractica3Ejercicio2.ipynb

Notas:
 * Para realizar el muestreo $z$ permitiendo la retropropagación del error se usa el "truco de reparametrización": no muestrear directamente la distribución gaussiana obtenida (eso rompería el grafo de cómputo), sino la distribución gaussiana $\hat{z} \sim \mathcal{N}(0, 1)$ y aprovechar la propiedad de las gaussianas para convertir el muestreo de una distribución gaussiana en el de otra para que (una vez simplificado aprovechando que se muestrea la normal) quede  $z =  \hat{z} \sigma + \mu$.
 * Es más fácil (y equivalente) que el encoder genere el logaritmo de la varianza en lugar de la varianza. Así se evita la posibilidad de intentar computar un $log(0)$. Obviamente hay que ajustar las fórmulas que se implementan de manera acorde.
 * Aunque PyTorch implementa diversas funciones para calcular la divergencia KL, ninguna está directamente pensada para este uso y su aplicación no es directa, así que es más fácil simplemente implementarla. Hay que recordar usar los operadores de PyTorch para que se genere el grafo de cómputo sin problema y que la pérdida de un lote debe ser la media de las pérdidas de los ejemplos que lo forman.
 * El valor $\alpha$ para conseguir un entrenamiento apropiado puede ser muy bajo según el conjunto de datos al que se aplique (incluso menos de 0.01).

In [15]:
# Import libraries.
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.utils.data as data_utils
import torch.nn.functional as F
from tqdm import tqdm

In [16]:
# Set device and define transformations.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.ToTensor()

# Load MNIST dataset.
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Define the sizes for training and validation datasets.
train_size = 0.8 * train_dataset.data.size(0)
val_size = 0.2 * train_dataset.data.size(0)

# Split the training dataset into training and validation sets.
train_dataset, val_dataset = data_utils.random_split(train_dataset, [int(train_size), int(val_size)])

# Create data loaders for training, validation, and testing datasets.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [17]:
class VAE(nn.Module):
    # Define the Variational Autoencoder (VAE) class.
    def __init__(self):
        super(VAE, self).__init__()

        # Define the encoder network.
        self.base_codifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.mean_layer = nn.Linear(64, 8)  # Mean layer for latent space.
        self.log_var_layer = nn.Linear(64, 8)  # Log variance layer

        # Define the decoder network.
        self.decoder = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid()
        )

    # Define the reparameterization trick.
    def reparameterize(self, mean, log_var):
        # Reparameterization trick to sample from the latent space.
        sigma = torch.exp(0.5 * log_var)
        # Noise sampled from a standard normal distribution.
        epsilon = torch.randn_like(sigma)
        # Return the sampled latent vector.
        z = mean + sigma * epsilon
        return z

    # Define the forward pass of the VAE.
    def forward(self, input_image):
        codified_image = self.base_codifier(input_image)  # Pass input through the encoder.
        mean = self.mean_layer(codified_image)  # Get the mean of the latent space.
        log_var = self.log_var_layer(codified_image)  # Get the log variance of the latent space.
        z = self.reparameterize(mean, log_var)  # Sample from the latent space.
        reconstructed_image = self.decoder(z)  # Decode the sampled latent vector.
        # Return the reconstructed image, mean, and log variance.
        return reconstructed_image.view(-1, 1, 28, 28), mean, log_var # .view(-1, 1, 28, 28) reshapes the output to match the original image dimensions (batch_size, channels, height, width).

In [18]:
def VAE_loss_function(reconstructed_image, input_image, mean, log_var, alpha=0.01):
    # Calculate the total loss for the VAE, which includes reconstruction loss and KL divergence.
    # Reconstruction + alpha * KL divergence losses summed over all elements and batch.
    loss_reconstruction = F.binary_cross_entropy(reconstructed_image, input_image, reduction='sum')  # Reconstruction loss (sum of errors of all pixels).

    # KL divergence loss (sum of the KL divergence for all elements in the batch).
    loss_kl_divergence = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp()) # mean.pow(2) = mu^2, log_var.exp() = sigma^2

    # Total loss is the sum of reconstruction loss and weighted KL divergence loss. (with alpha hiperparameter to control the weight of KL divergence).
    total_loss = loss_reconstruction + alpha * loss_kl_divergence

    sample_size = input_image.size(0)  # Get the batch size.
    return total_loss / sample_size  # Return the average (media) loss per sample in the batch.

In [19]:
def learning_loop_VAE(model, train_loader, val_loader, optimizer, alpha=0.01, num_epochs=10):
    train_historial_loss = []  # List to store the loss for each epoch.
    val_historial_loss = []  # List to store the loss for each epoch.

    for epoch in range(num_epochs):
        # Training phase
        model.train()  # Set the model to training mode.
        train_loss_accumulated = 0.0  # Initialize training loss for the epoch.
        loop_train = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}')  # Progress bar for training.

        for images, _ in loop_train: # Loop through the training data loader.
            images = images.to(device) # Move images to the specified device (CPU or GPU).

            optimizer.zero_grad()  # Zero the gradients for the optimizer.
            reconstructed_image, mean, log_var = model(images)  # Forward pass through the model.
            loss_train = VAE_loss_function(reconstructed_image, images, mean, log_var, alpha)  # Calculate the loss.
            loss_train.backward()  # Backpropagate the loss.
            optimizer.step()  # Update the model parameters.

            train_loss_accumulated += loss_train.item() * images.size(0)  # Accumulate the loss for the epoch.
            loop_train.set_postfix({'Loss': f'{train_loss_accumulated / len(train_loader.dataset):.4f}'})

        train_loss_media = train_loss_accumulated / len(train_loader.dataset)  # Calculate the average training loss for the epoch.
        train_historial_loss.append(train_loss_media)

        # Validation phase
        model.eval()  # Set the model to evaluation mode.
        val_loss_accumulated = 0.0  # Initialize validation loss for the epoch.
        loop_val = tqdm(val_loader, desc=f'Validation Epoch {epoch + 1}/{num_epochs}')  # Progress bar for validation.

        with torch.no_grad():  # Disable gradient calculation for validation.
            for images, _ in loop_val:  # Loop through the validation data loader.
                images = images.to(device)  # Move images to the specified device (CPU or GPU).

                reconstructed_image, mean, log_var = model(images)  # Forward pass through the model.
                loss_val = VAE_loss_function(reconstructed_image, images, mean, log_var, alpha)  # Calculate the loss.

                val_loss_accumulated += loss_val.item() * images.size(0)  # Accumulate the loss for the epoch.
                loop_val.set_postfix({'Loss': f'{val_loss_accumulated / len(val_loader.dataset):.4f}'})

        val_loss_media = val_loss_accumulated / len(val_loader.dataset)  # Calculate the average validation loss for the epoch.
        val_historial_loss.append(val_loss_media)

        print (f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {train_loss_media:.4f}, Val Loss: {val_loss_media:.4f}')

    return model, train_historial_loss, val_historial_loss  # Return the trained model and the loss history.


In [ ]:
model = VAE().to(device)  # Instantiate the VAE model and move it to the specified device.
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Define the optimizer (Adam) for training the model.

model, train_historial_loss, val_historial_loss = learning_loop_VAE(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    alpha=0.005, 
    num_epochs=10
)